# Live Inference on an Epoched Stream — Validation Notebook

Runs the **real online inference path** (`OnlinePreprocessor` + `LiveInferenceEngine`)
on a selected recording, then epochs the resulting probability stream by marker so
we can see, per decoder:

- the prediction time-course for **individual epochs**, and
- the **average prediction per marker** (the "does the inference make sense?" view).

**Method.** The whole recording is pushed through the online pipeline in micro-batches
with *persistent* causal filter state — exactly like `StreamWorker` — producing a
continuous 100 Hz probability stream. That stream is then cut into trigger-locked
epochs. No LSL / QThread is involved, so it is deterministic and fast, and filter
state stays continuous across the whole recording (no per-epoch warm-up artifact).

**Caveats**
1. **Leakage.** Running on the *training* recording yields optimistically-high
   probabilities. This is fine for a sanity check ("the decoder fires for the right
   marker"), but it is **not** a performance number — cross-validated AUC is.
2. **Single-timepoint model, swept over time.** Each decoder is a *spatial* classifier
   trained at one timepoint and applied at every 100 Hz sample; that sweep is what
   produces the per-epoch P(t) curve.
3. **Channel feed.** EEG is fed as **64 channels (EMG dropped) + trigger**, matching
   `scripts/replay_vhdr_to_lsl.py`. Offline computes `eeg_chunk_indices` on the
   EMG-*included* array, so there is an alignment assumption here (EMG last) — see the
   GH issue. Cell 4 asserts the alignment and fails loudly if it does not hold.
4. **Config must match the artifact.** Point `CONFIG_PATH` at the config the artifact
   was trained with (e.g. `debug_snapshots/experiment_config.yaml` for the cached
   debug artifact), or the recipe/markers/tasks will not line up.

*Validation only — not part of the app.*

In [ ]:
import sys
import re
from collections import Counter
from pathlib import Path

# Walk up to find src/, regardless of where Jupyter was launched from.
_search = Path().resolve()
REPO_ROOT = None
while _search != _search.parent:
    if (_search / "src" / "backend").exists():
        REPO_ROOT = _search
        if str(_search / "src") not in sys.path:
            sys.path.insert(0, str(_search / "src"))
        break
    _search = _search.parent

import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt
import numpy as np
import mne
mne.set_log_level("ERROR")

from backend.core.settings_manager import SettingsManager
from backend.online_phase.artifact_loader import load_decoder_pipeline_artifact
from backend.online_phase.online_preprocessor import OnlinePreprocessor
from backend.online_phase.live_inference import LiveInferenceEngine

print("repo root:", REPO_ROOT)

---
## Configuration

In [ ]:
# Use the config the cached artifact was trained with (6 decoders, fir lowpass).
CONFIG_PATH   = REPO_ROOT / "debug_snapshots" / "experiment_config.yaml"
ARTIFACT_PATH = REPO_ROOT / "debug_snapshots" / "models" / "decoder_pipeline.joblib"

# Default recording: functional localizer. Switch to "task" to run the harder block.
RECORDING_DIR = REPO_ROOT / "data" / "split" / "functional_localizer"
# RECORDING_DIR = REPO_ROOT / "data" / "split" / "task"

# Quick-run crop: process only the first N seconds of the recording.
# Set to None to run the full recording (slow — the localizer is tens of minutes).
MAX_SECONDS = 120

# Markers to epoch & display. None = auto: every decoder's positive label.
MARKERS_OF_INTEREST = None

# Mirror StreamWorker's default micro-batch so we exercise the same batching path.
BATCH_SIZE_SAMPLES = 40

---
## Load settings + trained artifact, build the live engine

In [ ]:
settings         = SettingsManager(CONFIG_PATH)
prepro_settings  = settings.get_preprocessing_params()
decoder_settings = settings.get_decoder_settings()
event_mapping    = settings.get_event_mapping()        # name -> trigger code
name_by_code     = {v: k for k, v in event_mapping.items()}

# Map each decoder to its positive marker (for the per-decoder individual-epoch view).
task_pos_marker = {t["name"]: t["pos_labels"][0] for t in decoder_settings["tasks"]}

# Auto-derive markers of interest from the decoders' positive labels if unset.
if MARKERS_OF_INTEREST is None:
    MARKERS_OF_INTEREST = list(dict.fromkeys(
        lbl for t in decoder_settings["tasks"] for lbl in t["pos_labels"]
    ))

artifact = load_decoder_pipeline_artifact(ARTIFACT_PATH)
preproc  = OnlinePreprocessor(prepro_settings, artifact.online_state)
engine   = LiveInferenceEngine(artifact.models, artifact.metadata)

eci        = list(artifact.online_state["eeg_chunk_indices"])
trained_tp = artifact.metadata.get("decoding_timepoint")

print("resample_filter_stage :", prepro_settings.get("resample_filter_stage"))
print("lowpass method        :", prepro_settings["lowpass"].get("method"))
print("tasks (models)        :", list(artifact.models.keys()))
print("markers of interest   :", MARKERS_OF_INTEREST)
print("feature_width         :", engine.feature_width)
print("preproc.n_channels    :", preproc.n_channels)
print("len(eeg_chunk_indices):", len(eci), "| max index:", max(eci))
print("trained timepoint     :", trained_tp)

assert engine.feature_width == preproc.n_channels, (
    "feature_width vs preproc channel-count mismatch"
)

---
## Load the recording (replay-style channels) and marker positions

Channels are handled exactly like `scripts/replay_vhdr_to_lsl.build_stream_matrix`:
keep EEG-typed channels, drop `EMG` → 64 EEG channels. Marker positions use the
**real** trigger codes parsed from the BrainVision `Stimulus/S xx` annotations.

In [ ]:
def find_vhdr(d):
    vhdrs = list(Path(d).glob("*.vhdr"))
    if not vhdrs:
        raise FileNotFoundError(f"No .vhdr file found in {d}")
    return vhdrs[0]

raw   = mne.io.read_raw_brainvision(find_vhdr(RECORDING_DIR), preload=True, verbose=False)
sfreq = float(raw.info["sfreq"])

# Replay-style channel handling: EEG-typed channels, EMG dropped -> 64 channels.
eeg_names = [raw.ch_names[i] for i in mne.pick_types(raw.info, eeg=True)]
if "EMG" in eeg_names:
    eeg_names.remove("EMG")
eeg = raw.copy().pick(eeg_names).get_data().T   # (n_times, 64), SI volts
n_times, n_fed = eeg.shape

# Optional crop for a quick smoke run.
if MAX_SECONDS is not None:
    n_keep = min(n_times, int(MAX_SECONDS * sfreq))
    eeg = eeg[:n_keep]
    n_times = eeg.shape[0]
    print(f"[crop] first {MAX_SECONDS}s -> {n_times} samples")

print(f"sfreq={sfreq:g} Hz | fed EEG channels={n_fed} | "
      f"samples={n_times} ({n_times / sfreq:.1f}s)")

# Channel-alignment parity check (see GH issue on EMG / eeg_chunk_indices).
assert max(eci) < n_fed, (
    f"eeg_chunk_indices max index {max(eci)} >= fed channel count {n_fed}. "
    "EMG / channel alignment mismatch between offline (EMG-included) and the "
    "64-channel replay feed. See the GH issue."
)

# Marker sample positions with REAL trigger codes: parse 'Stimulus/S 11' -> 11.
desc_to_code = {}
for d in set(raw.annotations.description):
    m = re.search(r"(\d+)\s*$", d)
    if m:
        desc_to_code[d] = int(m.group(1))
events, _ = mne.events_from_annotations(raw, event_id=desc_to_code, verbose=False)

codes_of_interest = {event_mapping[n] for n in MARKERS_OF_INTEREST}
markers = [(int(s), int(c)) for s, _, c in events
           if c in codes_of_interest and s < n_times]   # (sample, code)
print("epochable markers:",
      {name_by_code[c]: n for c, n in Counter(c for _, c in markers).items()})

---
## Run continuous live inference (micro-batched, persistent filter state)

`timestamps` carries the original sample index, so each 100 Hz output row maps back
to a raw sample — that mapping is what lets us epoch the prediction stream.

In [ ]:
preproc.reset_state()
sample_idx = np.arange(n_times)

feat_chunks, out_idx_chunks = [], []
for start in range(0, n_times, BATCH_SIZE_SAMPLES):
    sl = slice(start, start + BATCH_SIZE_SAMPLES)
    feats, out_ts = preproc.process_batch(eeg[sl], sample_idx[sl])
    if feats.shape[0]:
        feat_chunks.append(feats)
        out_idx_chunks.append(out_ts)

features    = np.vstack(feat_chunks)             # (n_out, n_channels) @ target rate
out_samples = np.concatenate(out_idx_chunks)     # original-sample index of each output row
preds       = engine.predict(features)           # {task: (n_out,)}
fs_out      = preproc.target_sfreq

print(f"output rows={features.shape[0]} @ {fs_out:g} Hz | tasks={list(preds.keys())}")

---
## Epoch the prediction stream by marker

Each marker's surrounding outputs are interpolated onto a common `[tmin, tmax]`
grid, giving `(n_epochs, n_times)` per task per marker.

In [ ]:
TMIN = prepro_settings["epochs"]["tmin"]
TMAX = prepro_settings["epochs"]["tmax"]
t_grid = np.arange(round(TMIN * fs_out), round(TMAX * fs_out) + 1) / fs_out

rel_time = out_samples / sfreq   # absolute output time in seconds

def epoch_stream(prob, marker_samples):
    rows = []
    for s in marker_samples:
        rt = rel_time - s / sfreq                       # output time relative to marker
        m = (rt >= TMIN - 0.05) & (rt <= TMAX + 0.05)
        if m.sum() < 2:
            continue
        rows.append(np.interp(t_grid, rt[m], prob[m]))
    return np.array(rows) if rows else np.empty((0, len(t_grid)))

epoched = {}   # epoched[task][marker_name] = (n_epochs, len(t_grid))
for task, prob in preds.items():
    epoched[task] = {}
    for name in MARKERS_OF_INTEREST:
        code_ = event_mapping[name]
        ms = [s for s, c in markers if c == code_]
        epoched[task][name] = epoch_stream(prob, ms)

# Epoch counts per marker (same for every task — driven by the markers, not the model).
print("epochs per marker:",
      {name: epoched[next(iter(preds))][name].shape[0] for name in MARKERS_OF_INTEREST})

---
## Individual epochs — every decoder on its own positive marker

One panel per decoder; each faint line is one trial's P(t) for that decoder's
target marker. Sanity: the mean (navy) should sit above 0.5 near the trained
timepoint if the decoder responds to its own stimulus.

In [ ]:
task_names = list(preds.keys())
ncols = 3
nrows = (len(task_names) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 3.6 * nrows), squeeze=False)

for idx, task in enumerate(task_names):
    ax = axes[idx // ncols][idx % ncols]
    marker = task_pos_marker.get(task)
    ep = epoched.get(task, {}).get(marker, np.empty((0, len(t_grid))))
    for row in ep:
        ax.plot(t_grid, row, color="steelblue", alpha=0.25, lw=0.8)
    if ep.shape[0]:
        ax.plot(t_grid, ep.mean(0), color="navy", lw=2.5, label=f"mean (n={ep.shape[0]})")
    ax.axvline(0, color="k", ls=":", lw=1)
    if trained_tp is not None:
        ax.axvline(trained_tp, color="crimson", ls="--", lw=1)
    ax.axhline(0.5, color="gray", lw=0.6)
    ax.set(title=f"{task} — '{marker}'", ylim=(0, 1),
           xlabel="time from marker (s)", ylabel="P(positive)")
    ax.legend(fontsize=7, loc="upper right")

for j in range(len(task_names), nrows * ncols):
    axes[j // ncols][j % ncols].axis("off")
plt.tight_layout(); plt.show()

---
## Average prediction per marker — the "does it make sense?" view

One panel per decoder, all markers overlaid. For `"<x> decoder"`, expect the
`<x>` curve to rise and peak near the trained timepoint while the others stay
near chance.

In [ ]:
colors = {"red": "crimson", "green": "green", "yellow": "goldenrod",
          "living_room": "purple", "bathroom": "teal", "kitchen": "saddlebrown"}

task_names = list(preds.keys())
ncols = 3
nrows = (len(task_names) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 3.6 * nrows), squeeze=False)

for idx, task in enumerate(task_names):
    ax = axes[idx // ncols][idx % ncols]
    for name in MARKERS_OF_INTEREST:
        ep = epoched[task][name]
        if ep.shape[0] == 0:
            continue
        mean = ep.mean(0)
        sem  = ep.std(0) / np.sqrt(ep.shape[0])
        c = colors.get(name)
        ax.plot(t_grid, mean, color=c, lw=1.8, label=f"{name} ({ep.shape[0]})")
        ax.fill_between(t_grid, mean - sem, mean + sem, color=c, alpha=0.15)
    ax.axvline(0, color="k", ls=":", lw=1)
    if trained_tp is not None:
        ax.axvline(trained_tp, color="black", ls="--", lw=1)
    ax.axhline(0.5, color="gray", lw=0.6)
    ax.set(title=task, ylim=(0, 1),
           xlabel="time from marker (s)", ylabel="P(positive)")
    ax.legend(fontsize=6, loc="upper right")

for j in range(len(task_names), nrows * ncols):
    axes[j // ncols][j % ncols].axis("off")
plt.tight_layout(); plt.show()

---
## Quick summary at the trained timepoint

Mean P(positive) per decoder × marker. The diagonal (decoder vs its own marker)
should be the largest entry in each row if the decoders are sensible.

In [ ]:
if trained_tp is not None:
    ti = int(np.argmin(np.abs(t_grid - trained_tp)))
    w  = max(9, max(len(n) for n in MARKERS_OF_INTEREST) + 1)
    print(f"Mean P(positive) at trained tp={trained_tp:.3f}s (grid {t_grid[ti]:.3f}s):\n")
    header = "task".ljust(22) + "".join(n.rjust(w) for n in MARKERS_OF_INTEREST)
    print(header)
    print("-" * len(header))
    for task in preds:
        row = task.ljust(22)
        for name in MARKERS_OF_INTEREST:
            ep = epoched[task][name]
            row += (f"{ep[:, ti].mean():.3f}" if ep.shape[0] else "n/a").rjust(w)
        print(row)
else:
    print("No trained timepoint in metadata; skipping summary.")